In [2]:
# Isso abre uma janela para você subir os arquivos do seu computador
from google.colab import files
uploaded = files.upload()

Saving Ask_12_2025-04_2026.xlsx to Ask_12_2025-04_2026.xlsx
Saving Kommo_2024-2025.xlsx to Kommo_2024-2025.xlsx


In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [4]:
df_kommo = pd.read_excel('Kommo_2024-2025.xlsx')
df_ask   = pd.read_excel('Ask_12_2025-04_2026.xlsx')

print("KOMMO — linhas e colunas:", df_kommo.shape)
print("ASK   — linhas e colunas:", df_ask.shape)

KOMMO — linhas e colunas: (16775, 8)
ASK   — linhas e colunas: (6455, 108)


In [5]:
# Primeiras 3 linhas do Kommo
print("=== KOMMO ===")
print(df_kommo.head(3))

print()

# Primeiras 3 linhas do ASK
print("=== ASK ===")
print(df_ask[['Data de criação','Pediu cotação?','⭐ Reservou','Agente','Última reserva - Valor']].head(3))

=== KOMMO ===
                               Etapa do lead  Venda          Data Criada  \
0                  Sem retorno - com cotação      0  31.12.2025 23:13:24   
1  Outros assuntos não relacionados a vendas      0  31.12.2025 21:50:41   
2             Sem retorno - mensagem inicial      0  31.12.2025 21:23:20   

  Data da Reserva    Origem Quantidade de pessoas      Campanhas  \
0             NaN  Meta Ads                   NaN  Facebook - N2   
1             NaN  Meta Ads                   NaN  Facebook - N2   
2             NaN  Meta Ads                   NaN  Facebook - N2   

  Telefone comercial (contato)  
0               '+554891812858  
1               '+554991703980  
2               '+555189175722  

=== ASK ===
    Data de criação Pediu cotação? ⭐ Reservou     Agente  \
0  30/04/2026 22:45            Não        Não       Robô   
1  23/03/2026 07:55            Sim        Sim  Alissiana   
2  30/04/2026 21:51            Sim        Não       Robô   

  Última reserva - Val

In [6]:
# Converter a coluna de data para formato que o Python entende
df_kommo['Data Criada'] = pd.to_datetime(
    df_kommo['Data Criada'],
    format='%d.%m.%Y %H:%M:%S',
    errors='coerce'
)

# Criar coluna só com o mês
df_kommo['mes'] = df_kommo['Data Criada'].dt.to_period('M')

print("Período Kommo:", df_kommo['Data Criada'].min().date(), "até", df_kommo['Data Criada'].max().date())

Período Kommo: 2024-12-01 até 2025-12-31


In [7]:
sazonalidade = df_kommo.groupby('mes').agg(
    leads            = ('Etapa do lead', 'count'),
    fechados         = ('Etapa do lead', lambda x: (x == 'Pago').sum())
).reset_index()

sazonalidade['conversao_%'] = (sazonalidade['fechados'] / sazonalidade['leads'] * 100).round(1)

print(sazonalidade.to_string(index=False))

    mes  leads  fechados  conversao_%
2024-12   1053        35          3.3
2025-01   1327        53          4.0
2025-02    984        26          2.6
2025-03   1287        61          4.7
2025-04   1563        60          3.8
2025-05   1476        57          3.9
2025-06   1537        62          4.0
2025-07   1680        56          3.3
2025-08   1235        38          3.1
2025-09    923        29          3.1
2025-10   1212        29          2.4
2025-11   1081        25          2.3
2025-12   1417        22          1.6


In [8]:
gargalos = df_kommo[df_kommo['Etapa do lead'] != 'Pago']['Etapa do lead'].value_counts()

print("=== Por que os leads não fecharam? ===")
print(gargalos.head(12))

=== Por que os leads não fecharam? ===
Etapa do lead
Sem retorno - com cotação                    6720
Sem retorno - mensagem inicial               2449
Cancelado                                    1814
Outros assuntos não relacionados a vendas    1649
Pesquisando                                  1446
Mínimo de Noites                              519
Valor                                         415
Não tinha disponibilidade para a data         414
Sem data definida                             383
Desistiu da viagem                            160
Comprado do concorrente                       129
Não tinha o que procurava - acomodação        113
Name: count, dtype: int64


In [9]:
# Converter data
df_ask['Data de criação'] = pd.to_datetime(df_ask['Data de criação'], dayfirst=True, errors='coerce')
df_ask['mes'] = df_ask['Data de criação'].dt.to_period('M')

# Conversão por agente
por_agente = df_ask.groupby('Agente').agg(
    contatos  = ('Agente', 'count'),
    reservas  = ('⭐ Reservou', lambda x: (x == 'Sim').sum())
).reset_index()

por_agente['conversao_%'] = (por_agente['reservas'] / por_agente['contatos'] * 100).round(1)

# Filtrar só quem tem volume relevante
por_agente = por_agente[por_agente['contatos'] >= 10].sort_values('conversao_%', ascending=False)

print(por_agente.to_string(index=False))

   Agente  contatos  reservas  conversao_%
    Tayna       523        33          6.3
Alissiana       460        22          4.8
  Tatiane       900        33          3.7
  Leandra      1029        27          2.6
     Robô      3517        19          0.5
Instagram        17         0          0.0


In [10]:
humanos = ['Tatiane', 'Leandra', 'Tayna', 'Alissiana']

robo   = df_ask[df_ask['Agente'] == 'Robô']
humano = df_ask[df_ask['Agente'].isin(humanos)]

conv_robo   = (robo['⭐ Reservou'] == 'Sim').sum() / len(robo) * 100
conv_humano = (humano['⭐ Reservou'] == 'Sim').sum() / len(humano) * 100

print(f"Robô  → {len(robo)} contatos | {(robo['⭐ Reservou']=='Sim').sum()} reservas | {conv_robo:.1f}% conversão")
print(f"Humanos → {len(humano)} contatos | {(humano['⭐ Reservou']=='Sim').sum()} reservas | {conv_humano:.1f}% conversão")
print(f"\nHumanos convertem {conv_humano/conv_robo:.1f}x mais que o robô")

Robô  → 3517 contatos | 19 reservas | 0.5% conversão
Humanos → 2912 contatos | 115 reservas | 3.9% conversão

Humanos convertem 7.3x mais que o robô


In [11]:
# Kommo
ticket_kommo = df_kommo[df_kommo['Venda'] > 0]['Venda']

# ASK — precisa limpar o formato "R$ 1.248,00"
df_ask['Valor_num'] = (
    df_ask['Última reserva - Valor']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('\xa0', '')        # espaço invisível
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_ask['Valor_num'] = pd.to_numeric(df_ask['Valor_num'], errors='coerce')
ticket_ask = df_ask[df_ask['Valor_num'] > 0]['Valor_num']

print(f"Ticket médio  — Kommo 2025:     R$ {ticket_kommo.mean():.0f}")
print(f"Ticket médio  — ASK Jan-Abr 2026: R$ {ticket_ask.mean():.0f}")
print(f"Ticket mediano — Kommo:  R$ {ticket_kommo.median():.0f}")
print(f"Ticket mediano — ASK:    R$ {ticket_ask.median():.0f}")

Ticket médio  — Kommo 2025:     R$ 1863
Ticket médio  — ASK Jan-Abr 2026: R$ 1248
Ticket mediano — Kommo:  R$ 1222
Ticket mediano — ASK:    R$ 787


In [12]:
com_utm    = df_ask['UTM Source'].notna().sum()
sem_utm    = df_ask['UTM Source'].isna().sum()
total      = len(df_ask)

print(f"Contatos COM UTM Source:  {com_utm} ({com_utm/total*100:.1f}%)")
print(f"Contatos SEM UTM Source:  {sem_utm} ({sem_utm/total*100:.1f}%)")
print()
print("Conclusão: sem UTM, ROAS é impossível de calcular.")

Contatos COM UTM Source:  71 (1.1%)
Contatos SEM UTM Source:  6384 (98.9%)

Conclusão: sem UTM, ROAS é impossível de calcular.


In [13]:
investimento_mensal = 4200
meses = 12
investimento_anual = investimento_mensal * meses

pago_origem = df_kommo[df_kommo['Origem'].isin(['Meta Ads', 'Google'])]
total_leads_pagos = len(pago_origem)
conversoes_pagas = (pago_origem['Etapa do lead'] == 'Pago').sum()
taxa_conv_paga = conversoes_pagas / total_leads_pagos * 100

cac_anuncio = investimento_anual / conversoes_pagas

print(f"Leads de anúncio pago:     {total_leads_pagos}")
print(f"Conversões:                {conversoes_pagas}")
print(f"Taxa de conversão:         {taxa_conv_paga:.1f}%")
print(f"Investimento anual:        R$ {investimento_anual:,.0f}")
print(f"CAC (anúncio):             R$ {cac_anuncio:,.0f} por reserva")
print(f"Ticket médio:              R$ 1.863")
print(f"Margem por reserva:        R$ {1863 - cac_anuncio:,.0f}")
print()
print("Nota: CAC real é maior — exclui ASK, gestor de tráfego e equipe de reservas.")

Leads de anúncio pago:     11142
Conversões:                195
Taxa de conversão:         1.8%
Investimento anual:        R$ 50,400
CAC (anúncio):             R$ 258 por reserva
Ticket médio:              R$ 1.863
Margem por reserva:        R$ 1,605

Nota: CAC real é maior — exclui ASK, gestor de tráfego e equipe de reservas.


In [14]:
campanhas = df_kommo.groupby('Campanhas').agg(
    leads       = ('Etapa do lead', 'count'),
    conversoes  = ('Etapa do lead', lambda x: (x == 'Pago').sum())
).reset_index()

campanhas['conv_%'] = (campanhas['conversoes'] / campanhas['leads'] * 100).round(1)
campanhas = campanhas[campanhas['leads'] >= 10].sort_values('leads', ascending=False)

print("=== CAMPANHAS — Volume e Conversão ===")
print(campanhas.to_string(index=False))

=== CAMPANHAS — Volume e Conversão ===
     Campanhas  leads  conversoes  conv_%
 Facebook - C2    858           1     0.1
Instagram - C1    598           4     0.7
      Facebook    380           3     0.8
 Facebook - N2    279           1     0.4
            C2    120           0     0.0
     Instagram    111           5     4.5
            C1     49           1     2.0
            c2     15           0     0.0
         GRUPO     14           0     0.0
            N2     10           0     0.0


In [15]:
# Exportar dados tratados como CSV
sazonalidade.to_csv('sazonalidade_kommo.csv', index=False)
por_agente.to_csv('conversao_por_agente.csv', index=False)
campanhas.to_csv('campanhas.csv', index=False)

# Fazer download dos arquivos
from google.colab import files
files.download('sazonalidade_kommo.csv')
files.download('conversao_por_agente.csv')
files.download('campanhas.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# Colunas de campanha no ASK
campanhas_ask = [
    '15% OFF - C9', '25%OFF - C8', 'CAMPANHA LINK PARA SITE',
    'CAMPANHA NATAL', 'CAMPANHA VERÃO', 'Insta - lista',
    'Inverno - C7', 'Meta', 'Outono - C1', 'Outono - C2',
    'Outono - C6'
]

resultados = []
for camp in campanhas_ask:
    leads      = (df_ask[camp] == 'Sim').sum()
    reservas   = df_ask[(df_ask[camp] == 'Sim') & (df_ask['⭐ Reservou'] == 'Sim')].shape[0]
    conv       = round(reservas / leads * 100, 1) if leads > 0 else 0
    resultados.append({'Campanha': camp, 'Leads': leads, 'Reservas': reservas, 'Conv_%': conv})

camp_ask = pd.DataFrame(resultados).sort_values('Leads', ascending=False)
print(camp_ask.to_string(index=False))

# Sazonalidade ASK (para exportar)
saz_ask = df_ask.groupby('mes').agg(
    contatos  = ('Pediu cotação?', 'count'),
    cotacoes  = ('Pediu cotação?', lambda x: (x == 'Sim').sum()),
    reservas  = ('⭐ Reservou', lambda x: (x == 'Sim').sum())
).reset_index()
saz_ask['mes'] = saz_ask['mes'].astype(str)
saz_ask['conv_%'] = (saz_ask['reservas'] / saz_ask['contatos'] * 100).round(1)
print()
print(saz_ask.to_string(index=False))

KeyError: '15% OFF - C9'

In [ ]:
# Mostrar todas as colunas que contêm "C9", "C8", "CAMP", "Insta", "Inverno", "Outono", "Meta"
import re
cols_camp = [c for c in df_ask.columns if any(x in c for x in ['C9','C8','CAMP','Insta','Inverno','Outono','Meta','FOLLOW'])]
print(cols_camp)

In [ ]:
# Colunas de campanha no ASK
campanhas_ask = [
    '15% OFF - C9 ', '25%OFF - C8', 'CAMPANHA LINK PARA SITE',
    'CAMPANHA NATAL', 'CAMPANHA VERÃO', 'Insta - lista ',
    'Inverno - C7', 'Meta', 'Outono - C1', 'Outono - C2',
    'Outono - C6'
]

resultados = []
for camp in campanhas_ask:
    leads      = (df_ask[camp] == 'Sim').sum()
    reservas   = df_ask[(df_ask[camp] == 'Sim') & (df_ask['⭐ Reservou'] == 'Sim')].shape[0]
    conv       = round(reservas / leads * 100, 1) if leads > 0 else 0
    resultados.append({'Campanha': camp, 'Leads': leads, 'Reservas': reservas, 'Conv_%': conv})

camp_ask = pd.DataFrame(resultados).sort_values('Leads', ascending=False)
print(camp_ask.to_string(index=False))

# Sazonalidade ASK (para exportar)
saz_ask = df_ask.groupby('mes').agg(
    contatos  = ('Pediu cotação?', 'count'),
    cotacoes  = ('Pediu cotação?', lambda x: (x == 'Sim').sum()),
    reservas  = ('⭐ Reservou', lambda x: (x == 'Sim').sum())
).reset_index()
saz_ask['mes'] = saz_ask['mes'].astype(str)
saz_ask['conv_%'] = (saz_ask['reservas'] / saz_ask['contatos'] * 100).round(1)
print()
print(saz_ask.to_string(index=False))

In [ ]:
resultados_funil = []
for camp in campanhas_ask:
    total      = (df_ask[camp] == 'Sim').sum()
    if total == 0:
        continue
    cotacao    = df_ask[(df_ask[camp] == 'Sim') & (df_ask['Pediu cotação?'] == 'Sim')].shape[0]
    clicou     = df_ask[(df_ask[camp] == 'Sim') & (df_ask['Clicou em "Reservar agora"'] == 'Sim')].shape[0]
    reservou   = df_ask[(df_ask[camp] == 'Sim') & (df_ask['⭐ Reservou'] == 'Sim')].shape[0]

    resultados_funil.append({
        'Campanha'       : camp.strip(),
        'Total leads'    : total,
        'Pediu cotação'  : cotacao,
        'Cot_%'          : round(cotacao / total * 100, 1),
        'Clicou reservar': clicou,
        'Click_%'        : round(clicou / total * 100, 1),
        'Reservou'       : reservou,
        'Reserva_%'      : round(reservou / total * 100, 1)
    })

funil_camp = pd.DataFrame(resultados_funil).sort_values('Total leads', ascending=False)
print(funil_camp.to_string(index=False))

In [ ]:
from google.colab import files

sazonalidade.to_csv('sazonalidade_kommo.csv', index=False)
saz_ask['mes'] = saz_ask['mes'].astype(str)
saz_ask.to_csv('sazonalidade_ask.csv', index=False)
camp_ask.to_csv('campanhas_ask.csv', index=False)
campanhas.to_csv('campanhas_kommo.csv', index=False)
funil_camp.to_csv('funil_campanhas_ask.csv', index=False)

for arquivo in ['sazonalidade_kommo.csv','sazonalidade_ask.csv',
                'campanhas_ask.csv','campanhas_kommo.csv','funil_campanhas_ask.csv']:
    files.download(arquivo)

In [ ]:
por_agente.to_csv('conversao_por_agente.csv', index=False)

from google.colab import files
files.download('conversao_por_agente.csv')